In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(train_df, test_df):
    """
    銀行顧客離脱データセットの前処理および特徴量エンジニアリングを行う関数
    """
    print("前処理を開始します...")
    
    # 1. 正解ラベルの分離とIDの退避
    y_train = train_df['Exited'] if 'Exited' in train_df.columns else None
    
    # 学習用とテスト用を区別するためのフラグを立てて一時的に結合
    train_features = train_df.drop(columns=['Exited']) if 'Exited' in train_df.columns else train_df.copy()
    test_features = test_df.copy()
    
    train_features['is_train'] = 1
    test_features['is_train'] = 0
    
    combined = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)
    
    # 2. 名字（Surname）の数（Frequency Encoding）
    # 同じ苗字がデータ中に何回登場するか（家族ルールの抽出）
    surname_counts = combined['Surname'].value_counts()
    combined['Surname_Freq'] = combined['Surname'].map(surname_counts)
    
    # 3. 製品数（NumOfProducts）の非線形性を捉えるフラグ
    # クロス集計で解約率が極端だった部分を明示的にフラグ化
    combined['Is_Product_2'] = (combined['NumOfProducts'] == 2).astype(int)
    combined['Is_High_Product'] = (combined['NumOfProducts'] >= 3).astype(int)
    
    # 4. 残高（Balance）に関する特徴量
    # 残高が0かどうかのフラグ
    combined['Is_Balance_Zero'] = (combined['Balance'] == 0).astype(int)
    # 年収に対する残高の割合（分母の0割りを防ぐため +1）
    combined['Balance_to_Salary_Ratio'] = combined['Balance'] / (combined['EstimatedSalary'] + 1)
    
    # 5. 年齢と信用スコアの掛け合わせ
    combined['CreditScore_by_Age'] = combined['CreditScore'] / combined['Age']
    
    # 6. 交互作用特徴量（国 × 製品数）
    # 「ドイツの製品数3」のような強力な組み合わせを文字列として結合
    combined['Geo_Product'] = combined['Geography'].astype(str) + "_" + combined['NumOfProducts'].astype(str)
    
    # 7. カテゴリ変数のエンコーディング（Label Encoding）
    # GBDTで扱えるように文字列を数値（0, 1, 2...）に変換
    cat_cols = ['Geography', 'Gender', 'Geo_Product']
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
    
    # 8. モデルの学習に不要なカラムの削除
    # 変換元のSurname、一意な値であるCustomerId、行識別用のidは削除（idは後で提出に使うため別途保持）
    drop_cols = ['id', 'CustomerId', 'Surname']
    combined = combined.drop(columns=[col for col in drop_cols if col in combined.columns])
    
    # 9. 再び train と test に分離
    X_train = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    X_test = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    print(f"前処理が完了しました。特徴量数: {X_train.shape[1]}")
    return X_train, y_train, X_test


In [6]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- 1. データの読み込み ---
# ※ パスは環境に合わせて変更してください
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")

# 提出用にテストセットの 'id' を保持
test_ids = test['id']

# --- 2. 前処理の実行 ---
X_train, y_train, X_test = preprocess_data(train, test)

# --- 3. クロスバリデーションの設定（Stratified K-Fold） ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# スコア記録用とテスト予測の格納用
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

# LightGBMのハイパーパラメータ
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'random_state': 42,
    'verbose': -1
}

# --- 4. 学習と予測のループ ---
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n--- Fold {fold + 1} の学習開始 ---")
    
    # データの分割
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
    
    # LightGBM専用のデータセット形式に変換
    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va)
    
    # モデルの訓練
    model = lgb.train(
        lgb_params,
        trn_data,
        num_boost_round=1000,
        valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # 検証データでの予測結果を記録（Out-of-Fold）
    oof_preds[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    
    # テストデータでの予測（各Foldのモデルの平均をとるため加算）
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / cv.n_splits

# --- 5. ローカルスコア（CVスコア）の算出 ---
cv_score = roc_auc_score(y_train, oof_preds)
print(f"\n====================================")
# CV（交差検証）のROC-AUCスコアを出力
print(f"全体の OOF ROC-AUC スコア: {cv_score:.5f}")
print(f"====================================")

# --- 6. 提出ファイルの作成 ---
submission = pd.DataFrame({
    'id': test_ids,
    'Exited': test_preds
})
submission.to_csv('submission.csv', index=False)
print("submission.csv を保存しました！")

前処理を開始します...
前処理が完了しました。特徴量数: 17

--- Fold 1 の学習開始 ---

--- Fold 2 の学習開始 ---

--- Fold 3 の学習開始 ---

--- Fold 4 の学習開始 ---

--- Fold 5 の学習開始 ---

全体の OOF ROC-AUC スコア: 0.93420


PermissionError: [Errno 13] Permission denied: 'submission.csv'